In [1]:
import torch
import os, json, argparse
import torchaudio
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
from ch_test import prepare_unique_sentences, LANG_MAP, LANG_MAP_INV
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [9]:
cname = "gen_config"

with open(f'{cname}.json', 'rt', encoding='utf-8') as f:
    config = json.load(f)


In [10]:
device = "cuda"

torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)
ecapa2 = torch.jit.load(model_file, map_location='cuda')
ecapa2.half()

model_name = config["model_name"]
output_bpath = os.path.join(config["output_bpath"], model_name)
dataspeech_stats_path = config["dataspeech_stats_path"]
dataspeech_stats_fname = config["dataspeech_stats_fname"]
test_sentence_path = config["test_sentence_path"]
test_sentence_fname = config["test_sentnece_fname"]
n_samples = config["n_samples"]
speaker_ref_path = config["speaker_ref_path"]

dataspeech_path = os.path.join(dataspeech_stats_path, dataspeech_stats_fname)
test_sentence_full_path = os.path.join(test_sentence_path, test_sentence_fname)

randdf = pd.read_csv(dataspeech_path, sep='\t')
unique_test_sentence_df = prepare_unique_sentences(test_sentence_full_path, k=n_samples)

unique_speakers = randdf['speaker_id'].unique()
output_data = []

In [4]:
speaker_to_embedding = {}
speaker_to_avg_similarity = {}
for sid, speaker in tqdm(enumerate(unique_speakers)):
    speaker_df = randdf[randdf['speaker_id'] == speaker]
    sample_ids = speaker_df['sample_id'].tolist()
    conditioning_paths = [os.path.join(speaker_ref_path, speaker, f"{audio}.wav") for audio in sample_ids]
    ref_embeddings = []
    for cp in conditioning_paths:
        waveform, _ = torchaudio.load(cp)
        embedding = ecapa2(waveform.to(device))
        ref_embeddings.append(embedding.cpu().numpy())
        
    #compute pariwise similarity between embeddings
    similarity_matrix = cosine_similarity(np.array(ref_embeddings).squeeze())
    #compute average similarity, i.e., average upper triangular matrix
    div = len(similarity_matrix) * (len(similarity_matrix) - 1) / 2
    avg_similarity = np.triu(similarity_matrix, k=1).sum()/div
    speaker_to_avg_similarity[speaker] = avg_similarity
    
    ref_embeddings_avg = np.vstack(ref_embeddings).squeeze().mean(axis=0)
    speaker_to_embedding[speaker] = ref_embeddings_avg

27it [00:44,  1.65s/it]


In [8]:
#avg similarity of speakers
np.mean([float(x) for x in speaker_to_avg_similarity.values()])

np.float64(0.7729870166631783)

In [27]:
#compute pairwise similarity between speakers
from scipy.spatial.distance import cosine
from itertools import combinations
from collections import defaultdict
import numpy as np

speaker_similarity = defaultdict(dict)
for speaker, sim in speaker_to_avg_similarity.items():
    speaker_similarity[speaker][speaker] = float(sim)

for speaker1, speaker2 in combinations(unique_speakers, 2):
    sim = 1 - cosine(speaker_to_embedding[speaker1], speaker_to_embedding[speaker2])
    speaker_similarity[speaker1][speaker2] = float(sim)
    speaker_similarity[speaker2][speaker1] = float(sim)    

speaker_to_sim_range = {}
for speaker, sim_dict in speaker_similarity.items():
    sim_values = list(sim_dict.values())
    speaker_to_sim_range[speaker] = (float(min(sim_values)), float(max(sim_values)))


D:\miniconda3\envs\coqui_tts\Lib\site-packages\scipy\spatial\distance.py:685: RuntimeWarning: overflow encountered in scalar multiply
  dist = 1.0 - uv / math.sqrt(uu * vv)


In [30]:
for sid, speaker in enumerate(unique_speakers):
    opath_speaker = os.path.join(output_bpath, speaker)
    orig_dialect = randdf[randdf['speaker_id'] == speaker]['dialect'].iloc[0]
    orig_dialect_tag = LANG_MAP_INV[orig_dialect]
    ref_embeddings_avg = speaker_to_embedding[speaker]
    for idx, row in tqdm(
            unique_test_sentence_df.iterrows(),
            total=len(unique_test_sentence_df),
            desc=f"Speaker {sid}/{len(unique_speakers)}: {speaker}"
    ):
        for dial_tag in LANG_MAP.keys():
            audio_file = os.path.join(opath_speaker, dial_tag, f'sent-{idx}.wav')
            if not os.path.exists(audio_file):
                print(f"Skipping {opath_speaker}-{dial_tag}-{idx}")
                continue
            waveform, _ = torchaudio.load(audio_file)
            sample_embedding = ecapa2(waveform.to(device)).squeeze()
            similarity = float(torch.nn.functional.cosine_similarity(torch.tensor(ref_embeddings_avg, device=device)[None, :], sample_embedding))
            
            ms, mx = speaker_to_sim_range[speaker]
            lam = (similarity - ms) / (mx - ms)
            lam = max(0, min(1, lam))
            print(f"Similarity {speaker}: {similarity:.4f}, Lambda: {lam:.4f}")

Speaker 0/27: 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72:   0%|          | 0/50 [00:00<?, ?it/s]

Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3586, Lambda: 0.6622
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3484, Lambda: 0.6433
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3796, Lambda: 0.7010
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3542, Lambda: 0.6541
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3867, Lambda: 0.7141
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3882, Lambda: 0.7168
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3611, Lambda: 0.6667


Speaker 0/27: 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72:   2%|▏         | 1/50 [00:05<04:09,  5.09s/it]

Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3601, Lambda: 0.6649
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3801, Lambda: 0.7019
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3687, Lambda: 0.6807
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3579, Lambda: 0.6609
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3547, Lambda: 0.6550
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3660, Lambda: 0.6758
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3491, Lambda: 0.6447
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3250, Lambda: 0.6000


Speaker 0/27: 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72:   4%|▍         | 2/50 [00:09<03:57,  4.94s/it]

Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3135, Lambda: 0.5788
Similarity 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72: 0.3779, Lambda: 0.6979


Speaker 0/27: 031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72:   4%|▍         | 2/50 [00:11<04:27,  5.57s/it]


KeyboardInterrupt: 